# Módulo 4 · Comparación Cruzada de Métodos

Proyecto: Análisis y Diseño de Algoritmos · Optimización de Portafolios

Este notebook consolida los resultados de los tres módulos anteriores (Markowitz, NSGA-II
y Programación Dinámica) leyendo los archivos JSON generados por cada uno, y construye una
tabla comparativa y un ranking final por Sharpe Ratio.

**Requisito:** ejecutar previamente, en este mismo entorno (mismo runtime de Colab o
mismo directorio local), los notebooks 1, 2 y 3 para generar `resultados_m1.json`,
`resultados_m2.json` y `resultados_m3.json`.

In [ ]:
import json
import numpy as np
import pandas as pd
import plotly.graph_objects as go

## Carga de resultados de los módulos previos

In [ ]:
with open('resultados_m1.json') as f:
    m1 = json.load(f)
with open('resultados_m2.json') as f:
    m2 = json.load(f)
with open('resultados_m3.json') as f:
    m3 = json.load(f)

print("Modulos cargados correctamente: M1 (Markowitz), M2 (NSGA-II), M3 (Programacion Dinamica)")

Modulos cargados correctamente: M1 (Markowitz), M2 (NSGA-II), M3 (Programacion Dinamica)


## Reconstrucción de las curvas de riqueza

Se reconstruyen las 7 curvas de riqueza a comparar: 3 de Markowitz/Equiponderado,
3 portafolios representativos de NSGA-II, y 1 de Programación Dinámica.

In [ ]:
def dict_a_serie(d, usar_fechas=True):
    if usar_fechas:
        idx = pd.to_datetime(d['fechas'])
    else:
        idx = d.get('periodos', list(range(len(d['valores']))))
    return pd.Series(d['valores'], index=idx)

curvas = {
    'Markowitz - Buy & Hold': dict_a_serie(m1['wealth_bh']),
    'Markowitz - Rebalanceado': dict_a_serie(m1['wealth_mkw']),
    'Equiponderado': dict_a_serie(m1['wealth_eq']),
    'NSGA-II - Conservador': dict_a_serie(m2['wealth_conservador']),
    'NSGA-II - Balanceado': dict_a_serie(m2['wealth_balanceado']),
    'NSGA-II - Agresivo': dict_a_serie(m2['wealth_agresivo']),
    'Programacion Dinamica': dict_a_serie(m3['wealth_dp'], usar_fechas=False),
}

for nombre, serie in curvas.items():
    print(f"{nombre}: {len(serie)} observaciones")

Markowitz - Buy & Hold: 2559 observaciones
Markowitz - Rebalanceado: 2559 observaciones
Equiponderado: 2559 observaciones
NSGA-II - Conservador: 2559 observaciones
NSGA-II - Balanceado: 2559 observaciones
NSGA-II - Agresivo: 2559 observaciones
Programacion Dinamica: 12 observaciones


## Cálculo de métricas comparativas

Nota: la curva de Programación Dinámica usa periodicidad **mensual** (12 periodos/año),
mientras que las demás usan periodicidad **diaria** (252 sesiones/año). La tasa libre de
riesgo se asume en 0% para el cálculo del Sharpe Ratio.

In [ ]:
def metricas_desempeno(wealth_series, periodos_por_anio=252):
    w = pd.Series(wealth_series).astype(float)
    if len(w) < 2:
        return None
    rets = w.pct_change().dropna()
    retorno_total = w.iloc[-1] / w.iloc[0] - 1
    n_periodos = len(w)
    retorno_anual = (1 + retorno_total) ** (periodos_por_anio / max(n_periodos, 1)) - 1
    vol_anual = rets.std() * np.sqrt(periodos_por_anio)
    sharpe = retorno_anual / vol_anual if vol_anual and vol_anual > 0 else np.nan
    downside = rets[rets < 0].std()
    downside_anual = downside * np.sqrt(periodos_por_anio) if downside and not np.isnan(downside) else np.nan
    sortino = retorno_anual / downside_anual if downside_anual and downside_anual > 0 else np.nan
    drawdown = w / w.cummax() - 1
    max_dd = drawdown.min()
    return {
        'Retorno Total (%)': round(retorno_total * 100, 2),
        'Retorno Anualizado (%)': round(retorno_anual * 100, 2),
        'Volatilidad Anualizada (%)': round(vol_anual * 100, 2),
        'Sharpe Ratio': round(sharpe, 2) if pd.notna(sharpe) else np.nan,
        'Sortino Ratio': round(sortino, 2) if pd.notna(sortino) else np.nan,
        'Max Drawdown (%)': round(max_dd * 100, 2),
        'Riqueza Final (USD)': round(w.iloc[-1], 2),
    }

FRECUENCIA_ANUAL = {
    'Markowitz - Buy & Hold': 252, 'Markowitz - Rebalanceado': 252, 'Equiponderado': 252,
    'NSGA-II - Conservador': 252, 'NSGA-II - Balanceado': 252, 'NSGA-II - Agresivo': 252,
    'Programacion Dinamica': 12,
}

filas = []
for nombre, serie in curvas.items():
    met = metricas_desempeno(serie.values, periodos_por_anio=FRECUENCIA_ANUAL[nombre])
    if met:
        fila = {'Estrategia': nombre}
        fila.update(met)
        filas.append(fila)

df_comparacion = pd.DataFrame(filas)
df_comparacion

,Estrategia,Retorno Total (%),Retorno Anualizado (%),Volatilidad Anualizada (%),Sharpe Ratio,Sortino Ratio,Max Drawdown (%),Riqueza Final (USD)
0,Markowitz - Buy & Hold,124.89,8.31,28.00,0.30,0.42,-47.02,219585.72
1,Markowitz - Rebalanceado,124.89,8.31,28.00,0.30,0.42,-47.02,219585.72
2,Equiponderado,35.29,3.02,31.30,0.10,0.14,-56.92,136801.88
3,NSGA-II - Conservador,96.48,6.88,27.39,0.25,0.36,-48.56,192433.03
4,NSGA-II - Balanceado,125.36,8.33,27.96,0.30,0.42,-47.49,219917.83
5,NSGA-II - Agresivo,135.35,8.79,33.35,0.26,0.38,-59.25,227026.00
6,Programacion Dinamica,-13.05,-13.05,10.97,-1.19,-2.24,-15.37,90186.61


## Visualización comparativa (7 curvas superpuestas)

Todas las curvas se normalizan a base 100 en su punto de partida para permitir la
comparación directa independientemente de la periodicidad de cada método.

In [ ]:
fig_comp = go.Figure()
for nombre, serie in curvas.items():
    es_temporal = not isinstance(serie.index[0], (int, np.integer))
    x_vals = serie.index if es_temporal else list(range(len(serie)))
    y_norm = serie.values / serie.values[0] * 100
    fig_comp.add_trace(go.Scatter(x=x_vals, y=y_norm, name=nombre, mode='lines'))

fig_comp.update_layout(title='Evolucion de Riqueza Normalizada (Base 100) - 7 Estrategias',
                        xaxis_title='Tiempo', yaxis_title='Riqueza Normalizada')
fig_comp.show()

## Ranking final por Sharpe Ratio

In [ ]:
df_ranking = df_comparacion.sort_values('Sharpe Ratio', ascending=False).reset_index(drop=True)
df_ranking.index += 1

df_ranking.to_csv('ranking_final.csv', index=False)
print("Ranking guardado en ranking_final.csv")
print("=== Ranking Final por Sharpe Ratio ===")
df_ranking

Ranking guardado en ranking_final.csv
=== Ranking Final por Sharpe Ratio ===


,Estrategia,Retorno Total (%),Retorno Anualizado (%),Volatilidad Anualizada (%),Sharpe Ratio,Sortino Ratio,Max Drawdown (%),Riqueza Final (USD)
1,Markowitz - Buy & Hold,124.89,8.31,28.00,0.30,0.42,-47.02,219585.72
2,Markowitz - Rebalanceado,124.89,8.31,28.00,0.30,0.42,-47.02,219585.72
3,NSGA-II - Balanceado,125.36,8.33,27.96,0.30,0.42,-47.49,219917.83
4,NSGA-II - Agresivo,135.35,8.79,33.35,0.26,0.38,-59.25,227026.00
5,NSGA-II - Conservador,96.48,6.88,27.39,0.25,0.36,-48.56,192433.03
6,Equiponderado,35.29,3.02,31.30,0.10,0.14,-56.92,136801.88
7,Programacion Dinamica,-13.05,-13.05,10.97,-1.19,-2.24,-15.37,90186.61
